In [1]:
from TDScoringAgent import TDScoringAgent
from TDAgentBase import TDAgentEnum, TDAgentState
from intent_gate import create_intent_capsule, SecurityError
import time
import json

In [2]:
def test_case_1_valid_capsule():
    """
    Test Case 1: Valid Intent Capsule

    Flow:
    1. Create agent
    2. Orchestrator creates valid capsule (signed + not expired)
    3. Agent runs
    4. Intent Gate validates
    5. Agent executes
    """
    print("\n" + "=" * 70)
    print("TEST CASE 1: VALID INTENT CAPSULE")
    print("=" * 70 + "\n")

    # Step 1: Create agent
    secret_key = "dev_secret"
    scoring_agent = TDScoringAgent(
        agent_id=TDAgentEnum.SCORING_AGENT, secret_key=secret_key
    )
    print("✓ Step 1: Created TDScoringAgent")
    print()

    # Step 2: Create valid capsule (Orchestrator does this)
    trip_id = "TRIP-001"
    intent_capsule = create_intent_capsule(
        trip_id=trip_id,
        secret_key=secret_key,
        expires_at=time.time() + 60,  # Valid for 60 seconds
    )
    print("✓ Step 2: Orchestrator created Intent Capsule")
    print(f"  Trip ID: {trip_id}")
    print(f"  Signature: {intent_capsule['signature'][:20]}...")
    print()

    # Step 3: Create state
    trip_data: TDAgentState = {
        "trip_id": trip_id,
        "trip_context": {
            "distance_km": 40.3,
            "harsh_events": 3,
        },
        "intent_capsule": intent_capsule,
    }
    print("✓ Step 3: Created state with trip_context + Intent Capsule")
    print()

    # Step 4: Run agent
    try:
        result = scoring_agent.run(trip_data)
        print("✓ Step 4: Intent Gate validated capsule")
        print("  - Signature valid")
        print("  - Not expired")
        print("  - Gate PASSED")
        print()

        print("✓ Step 5: Agent executed successfully")
        print()

        print("RESULT:")
        print(json.dumps(result, indent=2))

    except SecurityError as e:
        print(f"✗ FAILED: {e}")

In [3]:
test_case_1_valid_capsule()


TEST CASE 1: VALID INTENT CAPSULE

✓ Step 1: Created TDScoringAgent

✓ Step 2: Orchestrator created Intent Capsule
  Trip ID: TRIP-001
  Signature: 285d35c1a63da9d59675...

✓ Step 3: Created state with trip_context + Intent Capsule

✓ Step 4: Intent Gate validated capsule
  - Signature valid
  - Not expired
  - Gate PASSED

✓ Step 5: Agent executed successfully

RESULT:
{
  "agent": "scoring_agent",
  "trip_id": "TRIP-001",
  "timestamp": "2026-03-25T22:46:36.119210Z",
  "status": "SUCCESS",
  "output": {
    "trip_score": 85,
    "pings_count": 0,
    "harsh_events_count": 3
  }
}


In [ ]:
# TEST CASE 2: EXPIRED INTENT CAPSULE


def test_case_2_expired_capsule():
    """
    Test Case 2: Expired Intent Capsule

    Flow:
    1. Create agent
    2. Orchestrator creates capsule that is ALREADY EXPIRED
    3. Agent tries to run
    4. Intent Gate checks expiration
    5. Gate REJECTS (expired)
    """
    print("\n" + "=" * 70)
    print("TEST CASE 2: EXPIRED INTENT CAPSULE")
    print("=" * 70 + "\n")

    # Step 1: Create agent
    secret_key = "dev_secret"
    scoring_agent = TDScoringAgent(
        agent_id=TDAgentEnum.SCORING_AGENT, secret_key=secret_key
    )
    print("✓ Step 1: Created TDScoringAgent")
    print()

    # Step 2: Create EXPIRED capsule (Orchestrator does this)
    trip_id = "TRIP-002"
    intent_capsule = create_intent_capsule(
        trip_id=trip_id,
        secret_key=secret_key,
        expires_at=time.time() - 10,  # Expired 10 seconds ago
    )
    print("✓ Step 2: Orchestrator created Intent Capsule (EXPIRED)")
    print(f"  Trip ID: {trip_id}")
    print(f"  Expired at: {int(intent_capsule['capsule']['expires_at'])}")
    print(f"  Current time: {int(time.time())}")
    print()

    # Step 3: Create state
    trip_data: TDAgentState = {
        "trip_id": trip_id,
        "trip_context": {
            "distance_km": 35.0,
            "harsh_events": 2,
        },
        "intent_capsule": intent_capsule,
    }
    print("✓ Step 3: Created state with trip_context + EXPIRED Intent Capsule")
    print()

    # Step 4: Try to run agent
    try:
        result = scoring_agent.run(trip_data)
        print("✗ FAILED: Agent should have been blocked!")

    except SecurityError as e:
        print("✓ Step 4: Intent Gate checked expiration")
        print("  - Capsule is EXPIRED")
        print("  - Gate REJECTED")
        print()
        print(f"Error raised: {e}")


# Run the test
test_case_2_expired_capsule()